# Unit 04 · The network

From a convolution written by hand to the project's network: two X-rays in, a 64³ cube of logits out.

| Part | After video | What you build | Suggested driver |
|---|---|---|---|
| A | 09 · Convolution from scratch | output sizes, a hand-rolled conv2d, the conv block | Noa / Adir / Noa |
| B | 10 · Encoder-decoder and U-Net | the 2D encoder, lifting to 3D, the 3D decoder, the whole network | Adir / Noa / Adir / Noa |

**Needs:** nothing from earlier units moved into `src/xray3d` (the pictures use the processed dataset v1 from
unit 02 if it's on disk, and a stand-in image if not). Runs on the laptop's CPU; no GPU needed.
Within a part, swap the keyboard on every TODO. Record it in `LOG.md`.

In [ ]:
# --- setup (written by Claude, no need to change it): the same cell works locally, on Colab and on Kaggle ---
import os, sys, subprocess
from pathlib import Path

UNIT = "04-network"
REPO_URL = "https://github.com/AdirBuskila/pelvis-xray-3d"
IN_CLOUD = "google.colab" in sys.modules or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_CLOUD:
    root = Path("/content" if "google.colab" in sys.modules else "/kaggle/working") / "pelvis-xray-3d"
    if not root.exists():  # Kaggle: turn Internet on in the notebook settings first
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:  # local: the repo is the first folder above this notebook that has a pyproject.toml
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root / "units" / UNIT)
for p in (root / "src", Path.cwd()):  # import xray3d (even before `pip install -e .`) and this unit's checks
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import xray3d
from xray3d import data, env, plotting
import test_unit as T

env.summary()

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from xray3d import torchdata

torch.manual_seed(0)
try:   # one real sample from the processed dataset (unit 02), for the pictures
    sid = torchdata.load_index()["subjects"][0]
    z = np.load(torchdata.dataset_root() / f"{sid}.npz")
    ap, lat, occ = z["ap"], z["lat"], z["occ"].astype(bool)
    ap_bone, lat_bone = z["ap_bone"], z["lat_bone"]
    print(f"sample {sid}: ap {ap.shape}, lat {lat.shape}, occ {occ.shape} ({occ.mean():.1%} of the cube is pelvis)")
except FileNotFoundError as e:
    print(e, "\n-> using a stand-in image; the shapes and checks don't need real data")
    yy, xx = np.mgrid[:128, :128]
    ap = lat = np.exp(-((yy - 64) ** 2 + (xx - 64) ** 2) / 900).astype(np.float32)
    ap_bone = lat_bone = (ap > 0.5).astype(np.float32)
    occ = None
plotting.show_row([ap, lat], ["AP (rows head->feet, cols R->L)", "LAT (rows head->feet, cols A->P)"], size=4)

---
## Part A · Convolution (after video 09)

A convolution layer slides a small kernel over the image; at every position it multiplies and sums.
Before writing one, the bookkeeping: **how big is the output?**

### TODO A1 · `conv_out_size`  ·  suggested driver: **Noa**

The output size along one axis, for an ordinary convolution and for a **transposed** one (the upsampling layer of
video 10). Every shape in the network follows from this formula, so you'll use it to plan the encoder below.

> **Nudge:** count the positions where the kernel fits: the padded input is n + 2·padding long and the kernel starts every `stride` pixels. For the transposed case, think of every input pixel stamping a k-wide kernel, `stride` apart: how far does the last stamp reach?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def conv_out_size(n, k, stride=1, padding=0, transposed=False):
    """Output size (along one axis) of a convolution layer.

    n          : input size (pixels)
    k          : kernel size
    stride     : step between two kernel positions
    padding    : zeros added on EACH side
    transposed : False -> nn.Conv2d / nn.Conv3d;  True -> nn.ConvTranspose2d / 3d (no output_padding)
    returns    : an int
    """
    raise NotImplementedError("TODO A1")

In [ ]:
T.check_conv_out_size(conv_out_size)

In [ ]:
# plan the encoder with it: a 3x3 conv with padding 1 keeps the size, stride 2 halves it
n = 128
for stage in range(1, 4):
    n = conv_out_size(n, 3, stride=2, padding=1)
    print(f"after stage {stage}: {n} x {n}")
# and the decoder: transposed conv, kernel 2, stride 2
n = 16
for level in range(2):
    n = conv_out_size(n, 2, stride=2, transposed=True)
    print(f"decoder up {level + 1}: {n}^3")

### TODO A2 · `conv2d_naive`  ·  suggested driver: **Adir**

What `nn.Conv2d` computes, written out: for every output position, take the window of the (padded) input under
the kernel and sum `window × kernel` over the input channels and the kernel's pixels; one sum per output channel.
Loops over output positions are fine (it's slow, that's the point of the comparison below).

> **Nudge:** pad first (a bigger zero tensor with x copied into its middle), then the window for output pixel (r, c) starts at input pixel (r·stride, c·stride). Broadcasting (B, 1, C_in, k, k) × (1, C_out, C_in, k, k) gives every output channel at once.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def conv2d_naive(x, w, b=None, stride=1, padding=0):
    """What nn.Conv2d computes, written as an explicit sliding window.

    x       : (B, C_in, H, W) tensor
    w       : (C_out, C_in, k, k) kernels
    b       : (C_out,) bias, or None
    returns : (B, C_out, H_out, W_out) with H_out = conv_out_size(H, k, stride, padding), same for W
    Don't call F.conv2d or nn.Conv2d inside: the check makes them fail.
    """
    raise NotImplementedError("TODO A2")

In [ ]:
T.check_conv2d_naive(conv2d_naive)

In [ ]:
# your convolution on the AP image, with the kernels of video 09
img = torch.from_numpy(torchdata.standardise(ap)).reshape(1, 1, 128, 128)
kernels = {
    "Sobel x (vertical edges)": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
    "Sobel y (horizontal edges)": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
    "blur": [[1, 2, 1], [2, 4, 2], [1, 2, 1]],
}
w = torch.tensor(list(kernels.values()), dtype=torch.float32).unsqueeze(1)   # (3 kernels, 1 channel, 3, 3)
w[2] /= 16
t0 = time.time(); mine = conv2d_naive(img, w, padding=1); t_mine = time.time() - t0
t0 = time.time(); theirs = F.conv2d(img, w, padding=1); t_torch = time.time() - t0
print(f"max difference {float((mine - theirs).abs().max()):.2e};  yours {t_mine * 1000:.0f} ms, PyTorch {t_torch * 1000:.1f} ms")
plotting.show_row([img[0, 0]] + [mine[0, i] for i in range(3)], ["AP"] + list(kernels), size=3.6)

**Weight sharing, in numbers.** A fully connected layer from a 128 × 128 image to a 128 × 128 feature map has a
weight for every (input pixel, output pixel) pair. A conv layer with 16 kernels of 3 × 3 has 16 × 9.

In [ ]:
n_fc = (128 * 128) * (128 * 128)                    # nn.Linear(128*128, 128*128, bias=False): too big to build for fun (1 GB)
conv = nn.Conv2d(1, 16, 3, padding=1, bias=False)
print(f"fully connected 128^2 -> 128^2 : {n_fc:>13,} weights")
print(f"conv 3x3, 1 -> 16 channels     : {sum(p.numel() for p in conv.parameters()):>13,} weights (and 16 feature maps)")

### TODO A3 · `ConvBlock`  ·  suggested driver: **Noa**

The building block of both halves of the network: **conv (3 × 3 or 3 × 3 × 3, padding 1) → InstanceNorm → ReLU**.
`dim` picks 2D (the encoder) or 3D (the decoder); `stride=2` makes it a downsampling block.
InstanceNorm rescales every channel of every sample to zero mean and unit variance, then applies a learned scale and
shift (`affine=True`). The conv has **no bias**: the normalisation right after it subtracts the mean anyway.

> **Nudge:** a module stores its layers in `__init__` (as attributes, so PyTorch finds their weights) and chains them in `forward`. Pick the classes with `dim`: nn.Conv2d / nn.Conv3d, nn.InstanceNorm2d / nn.InstanceNorm3d (with affine=True).

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
class ConvBlock(nn.Module):
    """conv (kernel 3, padding 1, the given stride, no bias) -> InstanceNorm (affine=True) -> ReLU.

    c_in, c_out : channels in and out
    stride      : 1 keeps the size, 2 halves it
    dim         : 2 -> Conv2d + InstanceNorm2d (images);  3 -> Conv3d + InstanceNorm3d (volumes)
    forward(x)  : (B, c_in, *size) -> (B, c_out, *size / stride)
    """

    def __init__(self, c_in, c_out, stride=1, dim=2):
        super().__init__()
        raise NotImplementedError("TODO A3")

    def forward(self, x):
        raise NotImplementedError("TODO A3")

In [ ]:
T.check_ConvBlock(ConvBlock)

**Explain-back (part A):** why does a 3 × 3 kernel with padding 1 keep the size, and why does stride 2 halve it
(and what happens with an odd size)? Why can a conv layer with 144 weights do a job a fully connected layer
needs 268 million for, and what does it give up? Why is "convolution" in deep learning not flipped?

**Part A done.** Move `conv_out_size`, `conv2d_naive` and `ConvBlock` into `src/xray3d/nets.py`.

---
## Part B · From 2D to 3D (after video 10)

The network of video 10, narrower so it trains on a laptop's CPU for the overfit test (unit 05) and in well under
an hour on a Colab/Kaggle T4. The **shapes** are the video's, with fewer channels:

| stage | shape (batch 4) |
|---|---|
| one view | 4 × 1 × 128 × 128 |
| encoder stages | 16 × 128², 32 × 64², 64 × 32², 128 × 16² |
| 1×1 heads (the maps that get lifted) | 4 × 64², 8 × 32², 16 × 16² |
| lift each, concat AP + LAT | 8 × 64³, 16 × 32³, 32 × 16³ |
| decoder at 16³ | 32 × 16³ |
| up to 32³, concat skip, 2 convs | 16 × 32³ |
| up to 64³, concat skip, 2 convs | 8 × 64³ |
| 1×1×1 head | **4 × 1 × 64 × 64 × 64** logits |

The encoder hands the decoder a map at **every** scale: the coarse one knows *what* is there, the fine ones
*where exactly* (video 10's skip connections, here crossing from 2D to 3D through the lifting).

### TODO B1 · `Encoder2D`  ·  suggested driver: **Adir**

One X-ray in, three feature maps out (fine to coarse). Stage 0 works at full size; every later stage halves the
size and grows the channels. The last three stages each go through a **1 × 1 conv** that squeezes their channels
(`out`): these are the maps that get lifted into 3D.

> **Nudge:** keep the stages in an nn.ModuleList and run them in order in forward, remembering every stage's output. Which outputs need a head? The last len(out) of them.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
class Encoder2D(nn.Module):
    """One X-ray -> feature maps at three scales (finest first), one per decoder level.

    stage 0      : ConvBlock(c_in, widths[0]), ConvBlock(widths[0], widths[0])                 full size
    stage s >= 1 : ConvBlock(widths[s-1], widths[s], stride=2), ConvBlock(widths[s], widths[s])  size halves
    heads        : the outputs of the LAST len(out) stages each go through an nn.Conv2d(.., out[i], kernel_size=1)
                   (with bias)
    Set self.out = tuple(out) (XrayTo3D reads it).

    forward(x) : (B, c_in, 128, 128) -> [(B, 4, 64, 64), (B, 8, 32, 32), (B, 16, 16, 16)] with the defaults
    """

    def __init__(self, c_in=1, widths=(16, 32, 64, 128), out=(4, 8, 16)):
        super().__init__()
        raise NotImplementedError("TODO B1")

    def forward(self, x):
        raise NotImplementedError("TODO B1")

In [ ]:
T.check_Encoder2D(Encoder2D)

In [ ]:
enc = Encoder2D()
for f in enc(torch.zeros(4, 1, 128, 128)):
    print(tuple(f.shape))
print(f"{sum(p.numel() for p in enc.parameters()):,} parameters in one encoder")

### TODO B2 · `lift_view`  ·  suggested driver: **Noa**

The step that makes this network 3D: every pixel of an X-ray is a **ray** through the body, so its feature vector
is copied to every voxel along that ray. AP rays run front to back (along A, array axis `j`), LAT rays run side to
side (along R, axis `i`). The image orientation is the dataset's (unit 01): rows run head → feet, AP columns run
the patient's Right → Left, LAT columns Anterior → Posterior. The cube's axes run the other way round for some of
these: index `i` grows towards **Right**, `j` towards **Anterior**, `k` towards **Superior** (head).

> **Nudge:** first turn the image into the cube's order: which image axis is k, and which one is i (AP) or j (LAT)? Which of them run backwards (flip)? Then insert the ray axis with a size-1 dimension and `expand` it (no copy needed).

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def lift_view(feat, view):
    """Back-project a view's 2D feature map into the cube: copy every pixel along its ray.

    feat    : (B, C, n, n) features in IMAGE orientation: rows run head -> feet; columns run the patient's
              Right -> Left (view "ap") or Anterior -> Posterior (view "lat")
    view    : "ap" (rays along A = array axis j) or "lat" (rays along R = array axis i)
    returns : (B, C, n, n, n) in the cube's (i, j, k) array order: i grows towards Right, j towards Anterior,
              k towards Superior (head)
    """
    raise NotImplementedError("TODO B2")

In [ ]:
T.check_lift_view(lift_view)

**Look at it.** Which voxels does **one** pixel light up? Below, the 16 × 16 grid of the coarsest feature map over each
image (one cell = one feature pixel), one cell marked, and the voxels of the 16³ cube it reaches.
Check with the anatomy: the AP cell is on the image's left (the patient's **right**), so its ray must sit at
large `i`; the LAT cell is on the image's right (the patient's **back**), so small `j`.

In [ ]:
from matplotlib.patches import Rectangle
n = 16
fig = plt.figure(figsize=(13, 10))
picks = [("ap", ap, (3, 2), "AP", ("R", "L")), ("lat", lat, (3, 12), "LAT", ("A", "P"))]
for col, (view, im, (r, c), name, (left, right)) in enumerate(picks):
    a = fig.add_subplot(2, 2, 1 + col)
    a.imshow(im, cmap="gray", extent=(0, n, n, 0))
    a.set_xticks(range(n + 1)); a.set_yticks(range(n + 1)); a.grid(color="w", alpha=0.25); a.tick_params(labelbottom=False, labelleft=False)
    a.add_patch(Rectangle((c, r), 1, 1, color="#ffb347", alpha=0.9))
    a.set_title(f"{name}: feature pixel (row {r}, col {c})")
    a.text(-0.3, n / 2, left, ha="right", va="center", fontsize=14); a.text(n + 0.3, n / 2, right, ha="left", va="center", fontsize=14)
    a.text(n / 2, 0.6, "head", ha="center", va="center", color="w"); a.text(n / 2, n - 0.6, "feet", ha="center", va="center", color="w")
    f = torch.zeros(1, 1, n, n); f[0, 0, r, c] = 1.0
    v = lift_view(f, view)[0, 0].numpy() > 0.5
    ijk = np.argwhere(v)
    a3 = fig.add_subplot(2, 2, 3 + col, projection="3d")
    a3.voxels(v, facecolors="#ffb347", edgecolor="k", linewidth=0.3)
    a3.set(xlim=(0, n), ylim=(0, n), zlim=(0, n), xlabel="i  -> Right", ylabel="j  -> Anterior", zlabel="k  -> head")
    a3.view_init(elev=18, azim=-60 if view == "ap" else -30)
    vals = lambda ax_: sorted({int(a_) for a_ in ijk[:, ax_]})
    say = lambda ax_, name_: f"every {name_}" if len(vals(ax_)) == n else f"{name_} = {vals(ax_)[0]}"
    a3.set_title(f"lit voxels: {say(0, 'i')}, {say(1, 'j')}, {say(2, 'k')}")
plt.tight_layout(); plt.show()

**Lifting both silhouettes.** Lift the bone-only images (`ap_bone`, `lat_bone`, unit 03's oracle silhouettes) and
keep the voxels that both rays call bone: a visual hull (unit 03), made by your `lift_view`. The orange line is the
true pelvis. It's close, but not exact: `lift_view` assumes **parallel rays** and an image that covers exactly the
cube (320 mm), while the real image is a perspective one whose field of view at the cube's centre is different
(printed below, from the dataset's geometry), and the near side of the pelvis is magnified more than the far side.
Unit 03 carved the hull with the real camera instead. Here the network has to learn that correction itself.

In [ ]:
def to_t(im, size=64):   # (128, 128) numpy -> (1, 1, size, size) tensor
    return F.interpolate(torch.from_numpy(np.ascontiguousarray(im, dtype=np.float32))[None, None], size=(size, size), mode="area")

hull = (lift_view(to_t(ap_bone > 0), "ap") > 0.5) & (lift_view(to_t(lat_bone > 0), "lat") > 0.5)
hull = hull[0, 0].numpy()
c = np.argwhere(occ).mean(0).round().astype(int) if occ is not None else np.array([32, 32, 32])
show = lambda p: p.T[::-1, ::-1]   # display: head up; patient's right on the viewer's left (coronal), front on the left (sagittal)
fig, ax = plt.subplots(1, 3, figsize=(14, 5))
for a, (name, sl) in zip(ax, [("coronal (j fixed)", np.s_[:, c[1], :]), ("sagittal (i fixed)", np.s_[c[0], :, :]), ("axial (k fixed)", np.s_[:, :, c[2]])]):
    a.imshow(show(hull[sl]), cmap="gray")
    if occ is not None:
        a.contour(show(occ[sl]).astype(float), levels=[0.5], colors="#ffb347", linewidths=1.2)
    a.set_title(name); a.axis("off")
plt.suptitle("lift(AP silhouette) AND lift(LAT silhouette), white; the true pelvis, orange"); plt.tight_layout(); plt.show()
if occ is not None:
    g = torchdata.load_index()["geometry"]
    fov = g["detector_px"] * g["pixel_mm"] * g["sid_mm"] / g["sdd_mm"]   # detector width shrunk back to the cube's centre
    print(f"the image covers {fov:.0f} mm at the cube's centre; the cube is {g['cube_mm']:.0f} mm")
    print(f"Dice of this naive hull with the truth: {2 * (hull & occ).sum() / (hull.sum() + occ.sum()):.3f}")

### TODO B3 · `Decoder3D`  ·  suggested driver: **Adir**

The 3D half of the U-Net. Start at the coarsest volume (16³), then twice: upsample ×2 with a **transposed
convolution** (kernel 2, stride 2: no overlapping stamps, so no checkerboard), **concatenate** the skip volume of that
size along the channels, and refine with two 3D conv blocks. A 1 × 1 × 1 conv turns the last features into one
logit per voxel. Its bias starts at **−4**: then the untrained net predicts p = sigmoid(−4) ≈ 0.018 everywhere,
about the fraction of the cube that is pelvis (unit 05 shows why that matters).

> **Nudge:** build the levels in the order forward will use them (coarse to fine) and keep them in nn.ModuleLists. At level l the concat has widths[l] + c_in[l] channels: that's the first ConvBlock's input.

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
class Decoder3D(nn.Module):
    """Fused 3D volumes at three scales -> one logit per voxel of the 64^3 cube.

    c_in   : channels of the input volumes, finest first: (8, 16, 32) for 64^3, 32^3, 16^3
    widths : the decoder's channels at each level, finest first: (8, 16, 32)

    coarsest level     : ConvBlock(c_in[-1], widths[-1], dim=3), ConvBlock(widths[-1], widths[-1], dim=3)
    each finer level l : nn.ConvTranspose3d(widths[l+1], widths[l], kernel_size=2, stride=2, bias=False),
                         nn.InstanceNorm3d(.., affine=True), nn.ReLU  (size doubles); then torch.cat with vols[l] along the channels;
                         then ConvBlock(widths[l] + c_in[l], widths[l], dim=3), ConvBlock(widths[l], widths[l], dim=3)
    head               : nn.Conv3d(widths[0], 1, kernel_size=1), bias initialised to -4.0 (nn.init.constant_)

    forward(vols) : vols = [finest, ..., coarsest], e.g. [(B, 8, 64, 64, 64), (B, 16, 32, 32, 32), (B, 32, 16, 16, 16)]
                    -> (B, 1, 64, 64, 64) logits (no sigmoid)
    """

    def __init__(self, c_in=(8, 16, 32), widths=(8, 16, 32)):
        super().__init__()
        raise NotImplementedError("TODO B3")

    def forward(self, vols):
        raise NotImplementedError("TODO B3")

In [ ]:
T.check_Decoder3D(Decoder3D)

### TODO B4 · `XrayTo3D`  ·  suggested driver: **Noa**

Put it together. One `Encoder2D` per view (own weights: an AP and a LAT image look nothing alike), each map lifted
along its own view's rays, the views fused at every scale (`"concat"`, as in the video, or `"sum"`), dropout on the
coarsest volume, and the decoder. `n_views=1` is the AP-only network of unit 06's experiment.

> **Nudge:** loop over the views: encoder i gets x[:, i:i+1] (keep the channel axis) and its maps go through lift_view with that view's name. Then fuse level by level. With concat, how many channels reach the decoder at each level?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
class XrayTo3D(nn.Module):
    """The project's network: 1 or 2 X-rays -> occupancy logits of the 64^3 cube.

    n_views : 1 -> x is (B, 1, 128, 128), the AP image;  2 -> x is (B, 2, 128, 128), AP then LAT
    fuse    : how the views meet at every scale: "concat" (stack the channels) or "sum" (add them)
    dropout : nn.Dropout3d(dropout) on the fused coarsest volume (unit 07 keeps it on at test time: MC dropout)

    pieces  : an nn.ModuleList of n_views Encoder2D() (views in the order ("ap", "lat")); lift_view at every
              scale; fuse; dropout; Decoder3D(c_in) with c_in = the encoder's out (sum) or out x n_views (concat)
    forward(x) -> (B, 1, 64, 64, 64) logits
    """

    def __init__(self, n_views=2, fuse="concat", dropout=0.1):
        super().__init__()
        raise NotImplementedError("TODO B4")

    def forward(self, x):
        raise NotImplementedError("TODO B4")

In [ ]:
T.check_XrayTo3D(XrayTo3D)

**The table, from your code.** The same pieces as inside `XrayTo3D`, called one by one on a dummy batch of 4
(the video's "your turn"). Compare every line with the table at the top of part B.

In [ ]:
x = torch.zeros(4, 2, 128, 128)
enc_ap, enc_lat = Encoder2D(), Encoder2D()
dec = Decoder3D(tuple(2 * c for c in enc_ap.out))
with torch.no_grad():
    f_ap, f_lat = enc_ap(x[:, :1]), enc_lat(x[:, 1:])
    v_ap = [lift_view(f, "ap") for f in f_ap]
    v_lat = [lift_view(f, "lat") for f in f_lat]
    fused = [torch.cat([a, b], dim=1) for a, b in zip(v_ap, v_lat)]
    y = dec(fused)
print(f"{'input (AP, LAT)':<28}{tuple(x.shape)}")
for f in f_ap:
    print(f"{'encoder map (per view)':<28}{tuple(f.shape)}")
for v in fused:
    print(f"{'lifted, AP + LAT concat':<28}{tuple(v.shape)}")
print(f"{'decoder output (logits)':<28}{tuple(y.shape)}")

In [ ]:
print("parameters:")
for nv, fuse in [(1, "concat"), (2, "concat"), (2, "sum")]:
    m = XrayTo3D(nv, fuse=fuse)
    print(f"  XrayTo3D({nv}, fuse='{fuse}'): {sum(p.numel() for p in m.parameters()):>9,}")
print(f"  for scale: one fully connected layer from 2 x 128^2 pixels to 64^3 voxels: {2 * 128 ** 2 * 64 ** 3:,}")

model = XrayTo3D(2).eval()
with torch.no_grad():
    for B in (1, 8):
        xb = torch.randn(B, 2, 128, 128)
        model(xb)                                   # warm-up
        t0 = time.time(); model(xb); dt = time.time() - t0
        print(f"forward pass on this CPU, batch {B}: {dt:.2f} s")

**The untrained network on a real pair.** It has never seen a pelvis: the output is the −4 prior plus noise.
Unit 05 trains it.

In [ ]:
xr = torch.from_numpy(torchdata.standardise(np.stack([ap, lat])))[None]     # (1, 2, 128, 128), standardised per image
torch.manual_seed(0)
with torch.no_grad():
    prob = torch.sigmoid(XrayTo3D(2).eval()(xr))[0, 0].numpy()
print(f"probabilities: min {prob.min():.3f}, mean {prob.mean():.3f}, max {prob.max():.3f}")
plotting.show_row([show(prob[:, c[1], :]), show(occ[:, c[1], :]) if occ is not None else show(prob[:, c[1], :] * 0)],
                  ["untrained: p(bone), coronal", "the truth"], vmin=0, vmax=1, size=4)

**Explain-back (part B):**
1. Point at the AP pixel of the patient's right hip in the image. Which `(i, j, k)` voxels does `lift_view` copy it to,
   and why every `j`?
2. Why one encoder per view, and not one encoder fed a 2-channel image (video 10's first option)? What would that
   network have to learn by itself?
3. Where does the network lose resolution, and what brings the detail back?
4. Why start the head's bias at −4 and not 0? (Rough numbers: what fraction of the cube is pelvis?)
5. `lift_view` assumes parallel rays and an image exactly as wide as the cube. Our DRRs are neither. Who fixes that?
6. With `fuse="concat"`, the 1-view and 2-view networks differ by more than the second encoder. By what?
   Does that matter for unit 06's "same capacity" rule? Would `"sum"` be fairer?

**Part B done.** Move `Encoder2D`, `lift_view`, `Decoder3D` and `XrayTo3D` into `src/xray3d/nets.py` (with
`ConvBlock` from part A above them; `import torch` and `import torch.nn as nn` are already at the top of the file).
Units 05–08 build the network from there. Then run pytest:

In [ ]:
!{sys.executable} -m pytest -q -rs .

Fill in `LOG.md` (driver per TODO, hints, explain-back answers, the parameter count and the forward time).
Next: **unit 05**, training it.